## Importing libraries and downloading data

In [1]:
from functions import *
import pandas as pd
import re

In [2]:
pd.set_option('display.max_colwidth', None)

In [3]:
download_gbif()

GBIF backbone taxonomy has been downloaded successfully.


In [4]:
download_ncbi(taxonkitpath="./taxonkit")

NCBI taxonomic data has been downloaded successfully.


## Loading and processing samples

In [5]:
gbif_dataset = load_gbif_samples("./Taxon.tsv")

In [6]:
ncbi_dataset = load_ncbi_samples("./NCBI_output/All.lineages.tsv")

## Preparing samples for the classifier

In [7]:
positive_matches = generate_positive_set(gbif_dataset, ncbi_dataset, 50000)

In [8]:
negative_matches = generate_negative_set(gbif_dataset, ncbi_dataset, 50000)

 samples 16667 out of 16667

In [9]:
full_training_set, relevant_features = prepare_data(positive_matches, negative_matches)

## Choose the best classifier

In [10]:
X_train, X_test, y_train, y_test = generate_training_test(full_training_set, relevant_features)

In [11]:
compare_models(X_train, X_test, y_train, y_test)

,model,accuracy,mae,precision,recall,f1,roc,run_time,tp,fp,tn,fn
0,XGBClassifier,0.983491,0.016509,0.980332,0.986933,0.983622,0.983475,0.03,14565,297,14804,196
1,RandomForestClassifier,0.982553,0.017447,0.978898,0.986533,0.982701,0.982535,0.11,14543,319,14798,202
2,KNeighborsClassifier,0.977463,0.022537,0.974687,0.980600,0.977635,0.977448,0.02,14480,382,14709,291
3,DecisionTreeClassifier,0.973779,0.026221,0.972797,0.975067,0.973930,0.973773,0.01,14453,409,14626,374
4,GradientBoostingClassifier,0.969694,0.030306,0.960651,0.979800,0.970131,0.969647,0.13,14260,602,14697,303
5,AdaBoostClassifier,0.954290,0.045710,0.946026,0.964000,0.954928,0.954245,0.03,14037,825,14460,540
6,MLP,0.953252,0.046748,0.958541,0.947933,0.953208,0.953276,0.32,14247,615,14219,781
7,SupportVectorMachine,0.939153,0.060847,0.938527,0.940467,0.939496,0.939147,0.77,13938,924,14107,893
8,Perceptron,0.734613,0.265387,0.952077,0.496667,0.652793,0.735717,0.0,14487,375,7450,7550
9,DummyClassifier_stratified,0.502579,0.497421,0.504839,0.507733,0.506282,0.502555,0.0,7392,7470,7616,7384


In [12]:
model = XGBClassifier(learning_rate=0.1,n_estimators=500, max_depth=9, n_jobs=-1, colsample_bytree = 1, subsample = 0.8)
model.fit(X_train, y_train, verbose=False)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None, colsample_bytree=1,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, gamma=None, gpu_id=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=9, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=500, n_jobs=-1, num_parallel_tree=None,
              predictor=None, random_state=None, ...)

## Match NCBI with GBIF dataset 

In [14]:
ncbi_arthropoda_2, gbif_arthropoda_2 = select_taxonomic_clade_2("Arthropoda", gbif_dataset, ncbi_dataset)

In [17]:
df_matched_test, df_unmatched_test = match_datasets_2(gbif_arthropoda_2, ncbi_arthropoda_2, model, relevant_features, 0.80)

CPU times: user 9h 5min 43s, sys: 47min 38s, total: 9h 53min 21s
Wall time: 3h 29min 23s


## Managing Synonyms

In [19]:
ncbi_synonyms = get_ncbi_synonyms("./NCBI_output/names.dmp")
gbif_synonyms = get_gbif_synonyms(gbif_dataset)

In [89]:
df_matched_2, df_unmatched_2 = filter_synonyms(df_matched_test, df_unmatched_test, ncbi_synonyms, gbif_synonyms, gbif_arthropoda_2, ncbi_arthropoda_2)

## Generate the taxonomical tree 

In [90]:
df_matched_3, df_unmatched_3 = manage_duplicated_branches(df_matched_2, df_unmatched_2)

In [ ]:
taxonomic_tree = geneate_taxonomic_tree(df_matched_3, df_unmatched_3)